# 二维球面湍流 (2D Turbulence on a Sphere)

这是一个纯 Python 实现的二维球面湍流模拟（Barotropic vorticity equation on the unit sphere）。
不需要 `shtns` 库，仅依赖 NumPy 和 SciPy。

### 核心特点：
1. **全向量化球谐变换**：通过 `np.einsum` 实现高效的 Analysis 和 Synthesis。
2. **高斯-勒让德网格**：在纬向上使用高斯-勒让德节点，确保积分精度。
3. **交互式动画**：使用 `matplotlib.animation` 展示演化过程。

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import irfft, rfft
from tqdm.notebook import tqdm
from dataclasses import dataclass
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib as mpl

# 允许存储较大的动画
mpl.rcParams['animation.embed_limit'] = 100

In [ ]:
def _compute_legendre_matrices(lmax, mu, sin_theta):
    """
    使用稳定的递推关系计算正交归一化的缔合勒让德函数 P_l^m(mu) 及其导数。
    """
    J = len(mu)
    M = L = lmax + 1
    P = np.zeros((J, M, L), dtype=np.float64)
    dP = np.zeros((J, M, L), dtype=np.float64)
    
    # 1. 基础值：P_00 = sqrt(1 / 4pi)
    P[:, 0, 0] = np.sqrt(1.0 / (4.0 * np.pi))
    
    # 2. 对角线递推：l = m
    for m in range(1, M):
        P[:, m, m] = -np.sqrt((2.0 * m + 1.0) / (2.0 * m)) * sin_theta * P[:, m-1, m-1]
        
    # 3. 次对角线递推：l = m + 1
    for m in range(M):
        if m + 1 < L:
            P[:, m, m+1] = mu * np.sqrt(2.0 * m + 3.0) * P[:, m, m]
            
    # 4. 三对角递推：l > m + 1
    for m in range(M):
        for l in range(m + 2, L):
            a_lm = np.sqrt((4.0 * l**2 - 1.0) / (l**2 - m**2))
            b_lm = np.sqrt(((2.0 * l + 1.0) * ((l - 1)**2 - m**2)) / ((2.0 * l - 3.0) * (l**2 - m**2)))
            P[:, m, l] = a_lm * mu * P[:, m, l-1] - b_lm * P[:, m, l-2]
            
    # 5. 计算导数 dP/dtheta
    for m in range(M):
        for l in range(m, L):
            if l > 0:
                c_lm = np.sqrt(((2.0 * l + 1.0) * (l**2 - m**2)) / (2.0 * l - 1.0))
                dP[:, m, l] = (l * mu * P[:, m, l] - c_lm * P[:, m, l-1]) / sin_theta
            else:
                dP[:, m, l] = 0.0

    m_idx = np.arange(M)[:, None]
    l_idx = np.arange(L)[None, :]
    valid = l_idx >= m_idx
    
    return P, dP, valid

In [ ]:
@dataclass
class SphericalHarmonicTransform:
    lmax: int
    nlat: int | None = None
    nlon: int | None = None

    def __post_init__(self) -> None:
        if self.nlat is None:
            self.nlat = self.lmax + 1
        if self.nlon is None:
            self.nlon = 2 * (self.lmax + 1)
        self.nlat = int(self.nlat)
        self.nlon = int(self.nlon)
        self.mu, self.w = np.polynomial.legendre.leggauss(self.nlat)
        self.theta = np.arccos(self.mu)
        self.sin_theta = np.sqrt(np.maximum(1.0 - self.mu**2, 1e-30))
        self.phi = 2.0 * np.pi * np.arange(self.nlon) / self.nlon
        M = L = self.lmax + 1
        J, K = self.nlat, self.nlon // 2 + 1
        self.M, self.L, self.J, self.K = M, L, J, K
        P, dP, self.valid = _compute_legendre_matrices(self.lmax, self.mu, self.sin_theta)
        self.P = np.ascontiguousarray(P, dtype=np.float64)
        self.dP = np.ascontiguousarray(dP, dtype=np.float64)
        self.Pw = np.ascontiguousarray(self.P * self.w[:, None, None], dtype=np.float64)
        ell = np.broadcast_to(np.arange(L, dtype=np.float64)[None, :], (M, L))
        self.lap = np.zeros((M, L), dtype=np.float64)
        self.lap[self.valid] = -ell[self.valid] * (ell[self.valid] + 1.0)
        self.inv_lap = np.zeros_like(self.lap)
        mask = self.lap != 0.0
        self.inv_lap[mask] = 1.0 / self.lap[mask]
        self.mvals = np.arange(M, dtype=np.float64)
        self.im = 1j * self.mvals[:, None]
        self._freq = np.zeros((J, K), dtype=np.complex128)
        self._lm = np.zeros((M, L), dtype=np.complex128)

    def analysis(self, field, out=None):
        if out is None:
            out = np.empty((self.M, self.L), dtype=np.complex128)
        F = rfft(field, axis=1)
        np.einsum('jml,jm->ml', self.Pw, F[:, : self.M], out=out, optimize=True)
        out *= (2.0 * np.pi / self.nlon)
        out[~self.valid] = 0.0
        return out

    def synthesis(self, a, out=None):
        self._freq.fill(0.0)
        np.einsum('jml,ml->jm', self.P, a, out=self._freq[:, : self.M], optimize=True)
        self._freq[:, : self.M] *= self.nlon
        field = irfft(self._freq, n=self.nlon, axis=1)
        if out is not None:
            out[...] = field
            return out
        return field

    def dphi(self, a, out=None):
        np.multiply(a, self.im, out=self._lm)
        self._lm[~self.valid] = 0.0
        return self.synthesis(self._lm, out=out)

    def dtheta(self, a, out=None):
        self._freq.fill(0.0)
        np.einsum('jml,ml->jm', self.dP, a, out=self._freq[:, : self.M], optimize=True)
        self._freq[:, : self.M] *= self.nlon
        field = irfft(self._freq, n=self.nlon, axis=1)
        if out is not None: out[...] = field; return out
        return field

    def apply_laplacian(self, a, out=None):
        if out is None: out = np.empty_like(a)
        np.multiply(a, self.lap, out=out)
        out[~self.valid] = 0.0
        return out

    def invert_laplacian(self, a, out=None):
        if out is None: out = np.empty_like(a)
        np.multiply(a, self.inv_lap, out=out)
        out[~self.valid] = 0.0
        out[0, 0] = 0.0
        return out

In [ ]:
@dataclass
class SphereTurbulence2D:
    sht: SphericalHarmonicTransform
    nu: float = 1.0e-7
    filter_alpha: float = 36.0
    filter_order: int = 8

    def __post_init__(self) -> None:
        l = np.arange(self.sht.L, dtype=np.float64)
        self.spec_filter = np.exp(-self.filter_alpha * (l / self.sht.lmax) ** self.filter_order)
        self.spec_filter[0] = 1.0
        self.init_slope = np.ones(self.sht.L, dtype=np.float64)
        mask = l > 0
        self.init_slope[mask] = l[mask] ** (-1.0 / 3.0)
        self._psi = np.zeros((self.sht.M, self.sht.L), dtype=np.complex128)
        self._diff = np.zeros((self.sht.M, self.sht.L), dtype=np.complex128)

    def filter_coeffs(self, a, out=None):
        if out is None: out = np.empty_like(a)
        np.multiply(a, self.spec_filter[None, :], out=out)
        out[~self.sht.valid] = 0.0
        return out

    def streamfunction_from_vorticity(self, zeta_lm, out=None):
        return self.sht.invert_laplacian(zeta_lm, out=out)

    def velocity_from_streamfunction(self, psi_lm):
        u_theta = self.sht.dphi(psi_lm) / self.sht.sin_theta[:, None]
        u_phi = -self.sht.dtheta(psi_lm)
        return u_theta, u_phi

    def rhs(self, zeta_lm):
        psi_lm = self.streamfunction_from_vorticity(zeta_lm, out=self._psi)
        u_theta, u_phi = self.velocity_from_streamfunction(psi_lm)
        dzeta_dtheta = self.sht.dtheta(zeta_lm)
        dzeta_dphi = self.sht.dphi(zeta_lm) / self.sht.sin_theta[:, None]
        adv = u_theta * dzeta_dtheta + u_phi * dzeta_dphi
        adv_lm = self.filter_coeffs(self.sht.analysis(adv))
        self.sht.apply_laplacian(zeta_lm, out=self._diff)
        self._diff *= self.nu
        return -adv_lm + self._diff

    def random_initial_vorticity(self, seed=42, amplitude=1.0):
        rng = np.random.default_rng(seed)
        grid = rng.standard_normal((self.sht.J, self.sht.nlon))
        zeta_lm = self.sht.analysis(grid)
        zeta_lm *= self.init_slope[None, :]
        zeta_lm = self.filter_coeffs(zeta_lm)
        zeta_lm[0, 0] = 0.0
        return amplitude * zeta_lm

    def kinetic_energy(self, zeta_lm):
        psi_lm = self.streamfunction_from_vorticity(zeta_lm)
        zeta = self.sht.synthesis(zeta_lm)
        psi = self.sht.synthesis(psi_lm)
        integrand = -0.5 * psi * zeta
        return float((2.0 * np.pi / self.sht.nlon) * np.sum(self.sht.w[:, None] * integrand))

    def step_rk4(self, zeta_lm, dt):
        k1 = self.rhs(zeta_lm)
        k2 = self.rhs(zeta_lm + 0.5 * dt * k1)
        k3 = self.rhs(zeta_lm + 0.5 * dt * k2)
        k4 = self.rhs(zeta_lm + dt * k3)
        out = zeta_lm + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)
        return self.filter_coeffs(out)

In [ ]:
# 参数设置
lmax = 63
nlat = lmax + 1
nlon = 2 * (lmax + 1)
dt = 1.0e-2
steps = 1000
save_every = 20
nu = 1.0e-7

# 初始化
sht = SphericalHarmonicTransform(lmax=lmax, nlat=nlat, nlon=nlon)
model = SphereTurbulence2D(sht=sht, nu=nu)
zeta_lm = model.random_initial_vorticity(seed=42, amplitude=1.0)

# 历史记录
history = []
energy = []

# 时间推进
for n in tqdm(range(steps + 1)):
    if n % save_every == 0:
        zeta = sht.synthesis(zeta_lm)
        history.append(zeta.copy())
        energy.append(model.kinetic_energy(zeta_lm))
    
    if n < steps:
        zeta_lm = model.step_rk4(zeta_lm, dt)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(np.arange(len(energy)) * save_every * dt, energy, lw=1.8)
plt.xlabel("time")
plt.ylabel("kinetic energy")
plt.title("Kinetic Energy Evolution")
plt.grid(True)
plt.show()

In [ ]:
lon_deg = np.degrees(sht.phi)
lat_deg = 90.0 - np.degrees(sht.theta)

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.pcolormesh(lon_deg, lat_deg, history[0], shading="auto", cmap="RdBu_r")
ax.set_xlabel("longitude (deg)")
ax.set_ylabel("latitude (deg)")
ax.set_title(f"Vorticity Evolution (step=0)")
plt.colorbar(im, ax=ax, label=r"$\zeta$")
plt.tight_layout()

def update(frame):
    im.set_array(history[frame].flatten())
    ax.set_title(f"Vorticity Evolution (step={frame * save_every})")
    return [im]

ani = FuncAnimation(fig, update, frames=len(history), interval=100, blit=True)
HTML(ani.to_jshtml())